In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet('data_sample/news_prices_full_processed.parquet')
print(df.shape)
# lose not so much data so let's drop rows with missing price data to have the cleaner dataset for the target creation
df = df.loc[(df['prev_day_price'].notna()) & (df['curr_day_price'].notna()) & (df['next_day_price'].notna())]
print(df.shape)
df

(100428, 27)
(87995, 27)


,date,prev_date,future_date,title,description,maintext,ticker,prev_day_price,curr_day_price,next_day_price,...,emotion_fear,emotion_joy,emotion_sadness,emotion_disgust,emotion_surprise,emotion_neutral,date_day_of_week,prev_day_of_week,future_day_of_week,language
2,2019-01-22,2019-01-21,2019-01-23,UBS Warns on Client Activity After $13 Billion...,Withdrawals at the Zurich-based bank’s key glo...,(Bloomberg) -- UBS Group AG warned client acti...,C,63.12000,61.85000,62.13000,...,0.037095,0.002821,0.799185,0.043700,0.022429,0.071112,Tuesday,Monday,Wednesday,en
3,2019-08-01,2019-07-31,2019-08-02,Shopify Boosts Outlook on New Online Offerings,(Bloomberg) -- Shopify Inc. shares continued t...,(Bloomberg) -- Shopify Inc. shares continued t...,SHOP,317.88000,341.39001,332.19000,...,0.007578,0.095697,0.013748,0.009044,0.107670,0.752265,Thursday,Wednesday,Friday,en
4,2019-10-31,2019-10-30,2019-11-01,Wayfair Plunges as Forecast Miss Heightens Gro...,(Bloomberg) -- Wayfair Inc. plunged near its l...,(Bloomberg) -- Wayfair Inc. plunged near its l...,C,72.97000,71.86000,73.84000,...,0.444956,0.007151,0.185429,0.010559,0.080362,0.254296,Thursday,Wednesday,Friday,en
5,2019-02-08,2019-02-07,2019-02-09,Skyscrapers Made of Wood Are Making a Comeback,"Sidewalk Labs LLC, a unit of Google parent Alp...",(Bloomberg) -- More than a century after steel...,GOOGL,1098.70996,1095.06006,1095.01001,...,0.001772,0.026602,0.002590,0.008262,0.009297,0.944932,Friday,Thursday,Saturday,en
6,2019-01-24,2019-01-23,2019-01-25,Intel Sales Miss as Data-Center Demand Slows; ...,Revenue in the current period will be about $1...,(Bloomberg) -- Intel Corp. reported lower-than...,GOOGL,1075.56995,1073.90002,1090.98999,...,0.515582,0.006684,0.080193,0.018094,0.074352,0.293145,Thursday,Wednesday,Friday,en
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100423,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,AAPL,133.19000,130.84000,129.71001,...,0.025531,0.007261,0.062263,0.011118,0.510062,0.360592,Wednesday,Tuesday,Thursday,en
100424,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,CVX,93.13000,95.92000,95.00000,...,0.025531,0.007261,0.062263,0.011118,0.510062,0.360592,Wednesday,Tuesday,Thursday,en
100425,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,MRK,74.25000,75.54000,75.41000,...,0.025531,0.007261,0.062263,0.011118,0.510062,0.360592,Wednesday,Tuesday,Thursday,en
100426,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt alien...,You can ride giant creatures and companions ca...,Benzinga\nWarren Buffett's Berkshire Cuts Appl...,BRK,245.28000,370500.00000,245.25000,...,0.025531,0.007261,0.062263,0.011118,0.510062,0.360592,Wednesday,Tuesday,Thursday,en


### Percentage changes of prices (target creation)

In [3]:
# price-return features
df['curr_prev_per_change'] = (df['curr_day_price'] - df['prev_day_price']) / df['prev_day_price']
df['next_curr_per_change'] = (df['next_day_price'] - df['curr_day_price']) / df['curr_day_price'] # THIS IS OUR TARGET

# 1 if next-day up, 0 otherwise
df['curr_prev_pos_change'] = (df['curr_prev_per_change'] > 0).astype(int)
df['next_curr_pos_change'] = (df['next_curr_per_change'] > 0).astype(int)

In [4]:
df.head(3)

,date,prev_date,future_date,title,description,maintext,ticker,prev_day_price,curr_day_price,next_day_price,...,emotion_surprise,emotion_neutral,date_day_of_week,prev_day_of_week,future_day_of_week,language,curr_prev_per_change,next_curr_per_change,curr_prev_pos_change,next_curr_pos_change
2,2019-01-22,2019-01-21,2019-01-23,UBS Warns on Client Activity After $13 Billion...,Withdrawals at the Zurich-based bank’s key glo...,(Bloomberg) -- UBS Group AG warned client acti...,C,63.12,61.85000,62.13,...,0.022429,0.071112,Tuesday,Monday,Wednesday,en,-0.020120,0.004527,0,1
3,2019-08-01,2019-07-31,2019-08-02,Shopify Boosts Outlook on New Online Offerings,(Bloomberg) -- Shopify Inc. shares continued t...,(Bloomberg) -- Shopify Inc. shares continued t...,SHOP,317.88,341.39001,332.19,...,0.107670,0.752265,Thursday,Wednesday,Friday,en,0.073959,-0.026949,1,0
4,2019-10-31,2019-10-30,2019-11-01,Wayfair Plunges as Forecast Miss Heightens Gro...,(Bloomberg) -- Wayfair Inc. plunged near its l...,(Bloomberg) -- Wayfair Inc. plunged near its l...,C,72.97,71.86000,73.84,...,0.080362,0.254296,Thursday,Wednesday,Friday,en,-0.015212,0.027554,0,1


### Feature generation

In [5]:
# text-length / style signals
df['title_word_count'] = (df['title'].astype(str).str.split(' ').apply(lambda x: len([w for w in x if w.strip() != ""])))
df['description_word_count'] = (df['description'].astype(str).str.split(' ').apply(lambda x: len([w for w in x if w.strip() != ""])))
df['maintext_word_count'] = (df['maintext'].astype(str).str.split(' ').apply(lambda x: len([w for w in x if w.strip() != ""])))

### Combine texts

In [6]:
pd.set_option('display.max_colwidth', None)

In [7]:
df.sample(5)

date  prev_date future_date  \
66071 2022-09-08 2022-09-07  2022-09-09   
79167 2021-01-13 2021-01-12  2021-01-14   
56427 2022-02-07 2022-02-06  2022-02-08   
373   2019-07-30 2019-07-29  2019-07-31   
1778  2019-02-08 2019-02-07  2019-02-09   

                                                                                                title  \
66071                          Samsung's new Galaxy Buds 2 Pro drop to $155 with first major discount   
79167                                     Webflow raises $140M, pushing its valuation to $2.1 billion   
56427  Trilogy Multifamily Income & Growth Holdings Acquires Class A Multifamily Community in Chicago   
373                                   Capital One hack exposes data of 100 million American customers   
1778                                                               Sony Plans 100 Billion Yen Buyback   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          description  \
66071                                                                                                                                                                                                                                                                                                                                                                                                                                                                           Amazon knocks 33 percent off Samsung's Galaxy Buds 2 Pro, bringing them down to a record low of $155.   
79167  This morning Webflow, a software company that helps businesses build no-code websites, announced that it has raised a $140 million Series B. The round, led by returning investors Accel and Silversmith, comes after the startup raised $72 million in an August, 2019 Series A. The new funding values Webflow at more than $2.1 billion it said in a blog post that TechCrunch viewed before publication. Webflows offers a software that helps customers build websites without the need to write code; the company also offers hosting, and content-related capabilities.   
56427                                                                                                                                                                                                                                                                                                                          Trilogy Multifamily Income & Growth Holdings, a Regulation A+ bond offering sponsored by Trilogy Real Estate Group, announced today the acquisition of a 138-unit, Class A multifamily community located in Chicago's famed Logan Square neighborhood.   
373                                                                                                                                                                                                                                                                                                                            Contained in the exposed data was credit scores, credit limits, balances, payment history, and contact information. The company was hacked in March. Read more here. Read more...More about Tech, Mashable Video, Cybersecurity, Hack, and Capital One   
1778                                                                                                                                                                                                                                                                      The repurch

In [8]:
df['title + description + maintext'] = df['title'].astype(str) + ' ' + df['description'].astype(str) + ' ' + df['maintext'].astype(str)
df['title + description'] = df['title'].astype(str) + ' ' + df['description'].astype(str)

In [9]:
pd.set_option('display.max_colwidth', 75)

In [10]:
df

,date,prev_date,future_date,title,description,maintext,ticker,prev_day_price,curr_day_price,next_day_price,...,language,curr_prev_per_change,next_curr_per_change,curr_prev_pos_change,next_curr_pos_change,title_word_count,description_word_count,maintext_word_count,title + description + maintext,title + description
2,2019-01-22,2019-01-21,2019-01-23,UBS Warns on Client Activity After $13 Billion of Outflows,Withdrawals at the Zurich-based bank’s key global wealth management uni...,(Bloomberg) -- UBS Group AG warned client activity is still recovering ...,C,63.12000,61.85000,62.13000,...,en,-0.020120,0.004527,0,1,10,46,707,UBS Warns on Client Activity After $13 Billion of Outflows Withdrawals ...,UBS Warns on Client Activity After $13 Billion of Outflows Withdrawals ...
3,2019-08-01,2019-07-31,2019-08-02,Shopify Boosts Outlook on New Online Offerings,(Bloomberg) -- Shopify Inc. shares continued their rally early Thursday...,(Bloomberg) -- Shopify Inc. shares continued their rally early Thursday...,SHOP,317.88000,341.39001,332.19000,...,en,0.073959,-0.026949,1,0,7,42,459,Shopify Boosts Outlook on New Online Offerings (Bloomberg) -- Shopify I...,Shopify Boosts Outlook on New Online Offerings (Bloomberg) -- Shopify I...
4,2019-10-31,2019-10-30,2019-11-01,Wayfair Plunges as Forecast Miss Heightens Growth Concerns,(Bloomberg) -- Wayfair Inc. plunged near its lowest levels of the year ...,(Bloomberg) -- Wayfair Inc. plunged near its lowest levels of the year ...,C,72.97000,71.86000,73.84000,...,en,-0.015212,0.027554,0,1,8,51,315,Wayfair Plunges as Forecast Miss Heightens Growth Concerns (Bloomberg) ...,Wayfair Plunges as Forecast Miss Heightens Growth Concerns (Bloomberg) ...
5,2019-02-08,2019-02-07,2019-02-09,Skyscrapers Made of Wood Are Making a Comeback,"Sidewalk Labs LLC, a unit of Google parent Alphabet Inc., is planning t...",(Bloomberg) -- More than a century after steel and concrete became the ...,GOOGL,1098.70996,1095.06006,1095.01001,...,en,-0.003322,-0.000046,0,0,8,48,657,"Skyscrapers Made of Wood Are Making a Comeback Sidewalk Labs LLC, a uni...","Skyscrapers Made of Wood Are Making a Comeback Sidewalk Labs LLC, a uni..."
6,2019-01-24,2019-01-23,2019-01-25,Intel Sales Miss as Data-Center Demand Slows; Shares Drop,Revenue in the current period will be about $16 billion and profit will...,(Bloomberg) -- Intel Corp. reported lower-than-projected fourth-quarter...,GOOGL,1075.56995,1073.90002,1090.98999,...,en,-0.001553,0.015914,0,1,9,54,675,Intel Sales Miss as Data-Center Demand Slows; Shares Drop Revenue in th...,Intel Sales Miss as Data-Center Demand Slows; Shares Drop Revenue in th...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100423,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt aliens as pets,You can ride giant creatures and companions can help you while you're e...,Benzinga\nWarren Buffett's Berkshire Cuts Apple Stake And Buys These Dr...,AAPL,133.19000,130.84000,129.71001,...,en,-0.017644,-0.008636,0,0,10,13,2821,'No Man's Sky' update lets players adopt aliens as pets You can ride gi...,'No Man's Sky' update lets players adopt aliens as pets You can ride gi...
100424,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt aliens as pets,You can ride giant creatures and companions can help you while you're e...,Benzinga\nWarren Buffett's Berkshire Cuts Apple Stake And Buys These Dr...,CVX,93.13000,95.92000,95.00000,...,en,0.029958,-0.009591,1,0,10,13,2821,'No Man's Sky' update lets players adopt aliens as pets You can ride gi...,'No Man's Sky' update lets players adopt aliens as pets You can ride gi...
100425,2021-02-17,2021-02-16,2021-02-18,'No Man's Sky' update lets players adopt aliens as pets,You can ride giant creatures and companions can help you while you're e...,Benzinga\nWarren Buffett's Berkshire Cuts Apple Stake And Buys These Dr...,MRK,74.25000,75.54000,75.41000,...,en,0.017374,-0.001721,1,0,10,13,2821,'No Man's Sky' upda

In [11]:
df.to_parquet('data_sample/news_prices_full_processed_with_target.parquet', index=False)

### Notes and TODOs

**Notes:**

+ It does not make sense to add features that change dymanically just as prices do, like volume, since we predict future, and they would not be available at the time of prediction

**TODO:**

+ Add historical features about stocks, like historical volatility, market cap, etc.

+ Add features related to risk-free interest rates, overall market sentiment, etc.

+ Think about some lag features related to prices, volumes, and their changes